# JRN Multiplicative Compounding on rel-f1 + Entropy vs GBM Probe Architecture Comparison

This notebook demonstrates an experiment on the **rel-f1 (Formula 1)** relational dataset testing two core hypotheses about the **Join Reproduction Number (JRN)**:

- **Part A**: Tests whether JRN compounds multiplicatively along multi-hop FK join chains (i.e., chain JRN ≈ product of individual JRNs).
- **Part B**: Compares an entropy-based JRN proxy against GBM-probe JRN, and evaluates 4 architecture configurations (probe-guided, entropy-guided, uniform-mean, uniform-rich).

The experiment uses LightGBM probes to measure how much predictive signal each FK join adds to downstream tasks.

In [ ]:
import subprocess, sys
def _pip(*a): subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *a])

# Core packages (pre-installed on Colab, install locally to match Colab env)
if 'google.colab' not in sys.modules:
    _pip('numpy==2.0.2', 'pandas==2.2.2', 'scikit-learn==1.6.1', 'scipy==1.16.3',
         'matplotlib==3.10.0', 'lightgbm==4.6.0')

In [ ]:
import gc
import json
import math
import os
import time
import warnings
from collections import Counter, defaultdict
from typing import Any, Dict, List, Optional, Tuple

import lightgbm as lgb
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats
from sklearn.metrics import roc_auc_score, mean_absolute_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

warnings.filterwarnings('ignore', category=UserWarning)
warnings.filterwarnings('ignore', category=FutureWarning)

print('All imports loaded successfully.')

In [ ]:
GITHUB_DATA_URL = "https://raw.githubusercontent.com/AMGrobelnik/ai-invention-bc07ab-join-reproduction-number-epidemiology-in/main/experiment_iter4_jrn_multiplicat/demo/mini_demo_data.json"

def load_data():
    """Load mini demo data from GitHub URL with local fallback."""
    try:
        import urllib.request
        with urllib.request.urlopen(GITHUB_DATA_URL) as response:
            return json.loads(response.read().decode())
    except Exception:
        pass
    if os.path.exists("mini_demo_data.json"):
        with open("mini_demo_data.json") as f:
            return json.load(f)
    raise FileNotFoundError("Could not load mini_demo_data.json")

In [ ]:
data = load_data()
print(f"Loaded {len(data['datasets'][0]['examples'])} examples")
print(f"Metadata: {data['metadata'].get('source', 'N/A')}")

## Configuration

All tunable parameters for the experiment. Set to minimum values for quick demo execution.

In [ ]:
# ── Tunable Parameters ──────────────────────────────────────────────────
# Minimum values for quick demo; original values commented out

SEEDS = [42]                  # Original: [42, 123, 456]
N_ESTIMATORS_INDIVIDUAL = 10  # Original: 200
N_ESTIMATORS_ARCH = 10        # Original: 300
N_ESTIMATORS_LOCAL = 10       # Original: 100
MAX_DEPTH_INDIVIDUAL = 3      # Original: 6
MAX_DEPTH_ARCH = 3            # Original: 8
MAX_DEPTH_LOCAL = 3           # Original: 4
N_BINS_ENTROPY = 10           # Original: 10

# Task and join definitions
DRIVER_TASKS = ["rel-f1/driver-dnf", "rel-f1/driver-top3", "rel-f1/driver-position"]
ALL_TASKS = DRIVER_TASKS

FK_DEFS = [
    {"id": "FK0",  "source": "races",                  "fk_col": "circuitId",      "target": "circuits",     "pk_col": "circuitId"},
    {"id": "FK1",  "source": "constructor_standings",   "fk_col": "raceId",         "target": "races",        "pk_col": "raceId"},
    {"id": "FK2",  "source": "constructor_standings",   "fk_col": "constructorId",  "target": "constructors", "pk_col": "constructorId"},
    {"id": "FK3",  "source": "standings",               "fk_col": "raceId",         "target": "races",        "pk_col": "raceId"},
    {"id": "FK4",  "source": "standings",               "fk_col": "driverId",       "target": "drivers",      "pk_col": "driverId"},
    {"id": "FK5",  "source": "constructor_results",     "fk_col": "raceId",         "target": "races",        "pk_col": "raceId"},
    {"id": "FK6",  "source": "constructor_results",     "fk_col": "constructorId",  "target": "constructors", "pk_col": "constructorId"},
    {"id": "FK7",  "source": "qualifying",              "fk_col": "raceId",         "target": "races",        "pk_col": "raceId"},
    {"id": "FK8",  "source": "qualifying",              "fk_col": "driverId",       "target": "drivers",      "pk_col": "driverId"},
    {"id": "FK9",  "source": "qualifying",              "fk_col": "constructorId",  "target": "constructors", "pk_col": "constructorId"},
    {"id": "FK10", "source": "results",                 "fk_col": "raceId",         "target": "races",        "pk_col": "raceId"},
    {"id": "FK11", "source": "results",                 "fk_col": "driverId",       "target": "drivers",      "pk_col": "driverId"},
    {"id": "FK12", "source": "results",                 "fk_col": "constructorId",  "target": "constructors", "pk_col": "constructorId"},
]

CHAINS = [
    {"id": "chain_results_races_circuits",          "type": "linear", "joins": ["FK10", "FK0"], "desc": "results->races->circuits"},
    {"id": "chain_standings_races_circuits",         "type": "linear", "joins": ["FK3", "FK0"],  "desc": "standings->races->circuits"},
    {"id": "chain_qualifying_races_circuits",        "type": "linear", "joins": ["FK7", "FK0"],  "desc": "qualifying->races->circuits"},
    {"id": "chain_conresults_races_circuits",        "type": "linear", "joins": ["FK5", "FK0"],  "desc": "con_results->races->circuits"},
    {"id": "chain_constandings_races_circuits",      "type": "linear", "joins": ["FK1", "FK0"],  "desc": "con_standings->races->circuits"},
    {"id": "chain_results_standings_to_drivers",     "type": "convergent", "joins": ["FK11", "FK4"],  "desc": "results+standings->drivers"},
    {"id": "chain_results_qualifying_to_drivers",    "type": "convergent", "joins": ["FK11", "FK8"],  "desc": "results+qualifying->drivers"},
    {"id": "chain_results_conresults_to_constructors","type": "convergent","joins": ["FK12", "FK6"],  "desc": "results+con_results->constructors"},
    {"id": "chain_constandings_conresults_to_constructors","type": "convergent","joins": ["FK2", "FK6"],"desc": "con_standings+con_results->constructors"},
    {"id": "chain_drivers_results_races_circuits",   "type": "linear_3hop", "joins": ["FK11", "FK10", "FK0"], "desc": "drivers<-results->races->circuits"},
]

print(f"Config: {len(SEEDS)} seed(s), {len(FK_DEFS)} FK joins, {len(CHAINS)} chains, {len(ALL_TASKS)} tasks")

## Step 0: Reconstruct Relational Database

Parse the flat JSON examples back into relational tables (DataFrames), FK metadata, and task samples.

In [ ]:
def reconstruct_database(raw_data):
    """Reconstruct tables, FK metadata, and task samples from the flat JSON."""
    examples = raw_data["datasets"][0]["examples"]
    table_rows = defaultdict(list)
    table_pk_cols = {}
    fk_metadata = []
    task_samples = defaultdict(lambda: defaultdict(list))
    task_meta = {}

    for ex in examples:
        rt = ex["metadata_row_type"]
        if rt == "table_row":
            tname = ex["metadata_table_name"]
            pk_col = ex["metadata_primary_key_col"]
            pk_val = ex["metadata_primary_key_value"]
            table_pk_cols[tname] = pk_col
            features = json.loads(ex["input"])
            try:
                pk_v = int(pk_val)
            except (ValueError, TypeError):
                pk_v = pk_val
            features[pk_col] = pk_v
            table_rows[tname].append(features)
        elif rt == "fk_join_metadata":
            inp = json.loads(ex["input"])
            outp = json.loads(ex["output"])
            fk_metadata.append({**inp, **outp})
        elif rt == "task_sample":
            tname = ex["metadata_task_name"]
            fold = ex.get("metadata_fold_name", "train")
            features = json.loads(ex["input"])
            label = ex["output"]
            features["__label__"] = label
            features["__fold__"] = fold
            task_samples[tname][fold].append(features)
            if tname not in task_meta:
                task_meta[tname] = {
                    "task_type": ex.get("metadata_task_type", "regression"),
                    "entity_col": ex.get("metadata_entity_col"),
                    "entity_table": ex.get("metadata_entity_table"),
                    "target_col": ex.get("metadata_target_col"),
                }

    tables = {}
    for tname, rows in table_rows.items():
        df = pd.DataFrame(rows)
        pk_col = table_pk_cols[tname]
        if pk_col in df.columns:
            df[pk_col] = pd.to_numeric(df[pk_col], errors="coerce").fillna(0).astype(int)
        tables[tname] = df

    return tables, fk_metadata, task_samples, task_meta, table_pk_cols


tables, fk_metadata, task_samples, task_meta, table_pk_cols = reconstruct_database(data)

print(f"Reconstructed {len(tables)} tables:")
for tname, df in tables.items():
    print(f"  {tname}: {df.shape[0]} rows x {df.shape[1]} cols")
print(f"\nFK metadata: {len(fk_metadata)} joins")
print(f"Tasks: {list(task_meta.keys())}")

## Step 1: Feature Engineering Helpers

Functions to encode tables (label-encode strings, parse dates) and aggregate child features via FK relationships.

In [ ]:
def encode_table(df, pk_col):
    """Encode a table's features: label-encode strings, parse dates, keep numerics."""
    df = df.copy()
    drop_cols = []
    for col in df.columns:
        if col == pk_col:
            continue
        if df[col].dtype == object:
            sample = df[col].dropna().head(5)
            is_date = False
            for v in sample:
                if isinstance(v, str) and ("T" in v or "-" in v) and len(v) > 8:
                    try:
                        pd.to_datetime(v)
                        is_date = True
                        break
                    except (ValueError, TypeError):
                        pass
            if is_date:
                try:
                    dt = pd.to_datetime(df[col], errors="coerce")
                    df[f"{col}_year"] = dt.dt.year.fillna(0).astype(int)
                    df[f"{col}_month"] = dt.dt.month.fillna(0).astype(int)
                    df[f"{col}_day"] = dt.dt.day.fillna(0).astype(int)
                    drop_cols.append(col)
                    continue
                except Exception:
                    pass
            try:
                le = LabelEncoder()
                mask = df[col].notna()
                if mask.sum() > 0:
                    vals = df.loc[mask, col].astype(str)
                    df.loc[mask, col] = le.fit_transform(vals)
                    df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0)
                else:
                    df[col] = 0
            except Exception:
                drop_cols.append(col)
        else:
            df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0)
    if drop_cols:
        df = df.drop(columns=drop_cols, errors="ignore")
    for col in df.columns:
        if col == pk_col:
            continue
        if df[col].dtype == object:
            try:
                df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0)
            except Exception:
                df = df.drop(columns=[col], errors="ignore")
    return df


def _make_numeric_df(df):
    """Ensure all columns are numeric."""
    result = df.copy()
    to_drop = []
    for c in result.columns:
        if result[c].dtype not in [np.float64, np.float32, np.int64, np.int32, float, int]:
            try:
                result[c] = pd.to_numeric(result[c], errors="coerce").fillna(0)
            except Exception:
                to_drop.append(c)
    if to_drop:
        result = result.drop(columns=to_drop)
    return result.fillna(0)


def _aggregate_to_entity(source_df, entity_col, target_df, fk_col, pk_col,
                         agg_prefix, agg_funcs, tables, table_pk_cols,
                         source_table_name=""):
    """Compute aggregated features from a join, returning DataFrame keyed by entity_col."""
    NaN_RESULT = None
    if entity_col in source_df.columns:
        if target_df is not None and fk_col != entity_col:
            target_enc = encode_table(target_df, pk_col)
            target_numeric = target_enc.select_dtypes(include=[np.number]).columns.tolist()
            target_numeric = [c for c in target_numeric if c != pk_col and c != entity_col]
            if not target_numeric:
                return NaN_RESULT
            merge_right = target_enc[[pk_col] + target_numeric]
            source_with_target = source_df[[entity_col, fk_col]].drop_duplicates().merge(
                merge_right, left_on=fk_col, right_on=pk_col, how="left"
            )
            if pk_col != fk_col and pk_col in source_with_target.columns:
                source_with_target = source_with_target.drop(columns=[pk_col], errors="ignore")
            agg_cols = [c for c in target_numeric if c in source_with_target.columns]
            if not agg_cols:
                return NaN_RESULT
            grouped = source_with_target.groupby(entity_col)[agg_cols].agg(agg_funcs)
        else:
            source_enc = encode_table(source_df, entity_col)
            numeric_cols = source_enc.select_dtypes(include=[np.number]).columns.tolist()
            numeric_cols = [c for c in numeric_cols if c != entity_col]
            if not numeric_cols:
                return NaN_RESULT
            grouped = source_enc.groupby(entity_col)[numeric_cols].agg(agg_funcs)
        if isinstance(grouped.columns, pd.MultiIndex):
            grouped.columns = [f"agg_{agg_prefix}{col}_{func}" for col, func in grouped.columns]
        else:
            grouped.columns = [f"agg_{agg_prefix}{col}" for col in grouped.columns]
        return grouped.reset_index()

    # Bridge case
    source_pk = table_pk_cols.get(source_table_name, None)
    if source_pk is None:
        for src_name, src_pk in table_pk_cols.items():
            if src_pk in source_df.columns:
                source_pk = src_pk
                break
    if source_pk is None:
        return NaN_RESULT
    for bridge_name, bridge_df in tables.items():
        if entity_col in bridge_df.columns and source_pk in bridge_df.columns:
            try:
                target_enc = encode_table(target_df, pk_col) if target_df is not None else None
                if target_enc is None:
                    continue
                target_numeric = target_enc.select_dtypes(include=[np.number]).columns.tolist()
                target_numeric = [c for c in target_numeric if c != pk_col and c != entity_col]
                if not target_numeric:
                    continue
                source_with_target = source_df.merge(
                    target_enc[[pk_col] + target_numeric],
                    left_on=fk_col, right_on=pk_col, how="left"
                )
                bridge_link = bridge_df[[entity_col, source_pk]].drop_duplicates()
                merged = source_with_target.merge(bridge_link, on=source_pk, how="inner")
                agg_cols = [c for c in target_numeric if c in merged.columns]
                if not agg_cols or merged.empty:
                    continue
                grouped = merged.groupby(entity_col)[agg_cols].agg(agg_funcs)
                if isinstance(grouped.columns, pd.MultiIndex):
                    grouped.columns = [f"agg_{agg_prefix}{col}_{func}" for col, func in grouped.columns]
                else:
                    grouped.columns = [f"agg_{agg_prefix}{col}" for col in grouped.columns]
                return grouped.reset_index()
            except Exception:
                continue
    return NaN_RESULT


print("Feature engineering helpers loaded.")

## Step 2: Build Task-Level Datasets

Construct train/validation feature matrices and labels for each driver prediction task.

In [ ]:
def build_task_dataset(task_name, task_samples, task_meta, tables, table_pk_cols):
    """Build train/val feature matrices and labels for a task."""
    meta = task_meta[task_name]
    entity_col = meta["entity_col"]
    entity_table = meta["entity_table"]
    task_type = meta["task_type"]

    train_samples = task_samples[task_name].get("train", [])
    val_samples = task_samples[task_name].get("val", [])
    test_samples = task_samples[task_name].get("test", [])
    if len(val_samples) < 5 and len(test_samples) > 5:
        eval_samples = test_samples
    else:
        eval_samples = val_samples

    def samples_to_df(samples):
        rows, labels = [], []
        for s in samples:
            label = s["__label__"]
            if label == "masked":
                continue
            try:
                if task_type == "binary_classification":
                    label = int(float(label))
                else:
                    label = float(label)
            except (ValueError, TypeError):
                continue
            row = {k: v for k, v in s.items() if k not in ("__label__", "__fold__")}
            rows.append(row)
            labels.append(label)
        if not rows:
            return pd.DataFrame(), pd.Series(dtype=float)
        df = pd.DataFrame(rows)
        if entity_col and entity_col in df.columns:
            df[entity_col] = pd.to_numeric(df[entity_col], errors="coerce").fillna(0).astype(int)
        for col in list(df.columns):
            if col == entity_col:
                continue
            sample = df[col].dropna().head(3)
            is_date = any(isinstance(v, str) and "T" in v for v in sample)
            if is_date:
                try:
                    dt = pd.to_datetime(df[col], errors="coerce")
                    df[f"{col}_year"] = dt.dt.year.fillna(0).astype(int)
                    df[f"{col}_month"] = dt.dt.month.fillna(0).astype(int)
                    df = df.drop(columns=[col])
                except Exception:
                    df = df.drop(columns=[col], errors="ignore")
        return df, pd.Series(labels)

    X_train, y_train = samples_to_df(train_samples)
    X_val, y_val = samples_to_df(eval_samples)
    if X_train.empty or X_val.empty:
        return X_train, y_train, X_val, y_val, task_type

    if entity_table and entity_table in tables and entity_col:
        entity_df = tables[entity_table].copy()
        pk_col = table_pk_cols.get(entity_table, entity_col)
        entity_encoded = encode_table(entity_df, pk_col)
        entity_feats = [c for c in entity_encoded.columns if c != pk_col]
        if entity_feats and entity_col in X_train.columns:
            X_train = X_train.merge(entity_encoded, left_on=entity_col, right_on=pk_col, how="left")
            X_val = X_val.merge(entity_encoded, left_on=entity_col, right_on=pk_col, how="left")
            if pk_col != entity_col and pk_col in X_train.columns:
                X_train = X_train.drop(columns=[pk_col], errors="ignore")
                X_val = X_val.drop(columns=[pk_col], errors="ignore")

    for df in [X_train, X_val]:
        for col in df.columns:
            if df[col].dtype == object:
                try:
                    df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0)
                except Exception:
                    pass

    common_cols = sorted(set(X_train.columns) & set(X_val.columns))
    numeric_cols = []
    for c in common_cols:
        if X_train[c].dtype in [np.float64, np.float32, np.int64, np.int32, float, int]:
            numeric_cols.append(c)
        else:
            try:
                X_train[c] = pd.to_numeric(X_train[c], errors="coerce").fillna(0)
                X_val[c] = pd.to_numeric(X_val[c], errors="coerce").fillna(0)
                numeric_cols.append(c)
            except Exception:
                pass
    X_train = X_train[numeric_cols].fillna(0)
    X_val = X_val[numeric_cols].fillna(0)
    return X_train, y_train, X_val, y_val, task_type


print("Building task datasets...")
task_datasets = {}
for task_name in ALL_TASKS:
    if task_name not in task_meta:
        print(f"  Skipping {task_name}: not in data")
        continue
    X_train, y_train, X_val, y_val, task_type = build_task_dataset(
        task_name, task_samples, task_meta, tables, table_pk_cols
    )
    if X_train.empty or X_val.empty:
        print(f"  Skipping {task_name}: insufficient data")
        continue
    task_datasets[task_name] = {
        "X_train": X_train, "y_train": y_train,
        "X_val": X_val, "y_val": y_val,
        "task_type": task_type,
        "entity_col": task_meta[task_name]["entity_col"],
    }
    print(f"  {task_name}: train={X_train.shape}, val={X_val.shape}, type={task_type}")

## Step 3: JRN Computation via GBM Probes

Train LightGBM models with and without augmented features from each FK join to measure the **Join Reproduction Number (JRN)** = augmented_score / baseline_score.

In [ ]:
def train_eval_lgbm(X_train, y_train, X_val, y_val, task_type, seed,
                    n_estimators=None, max_depth=None):
    """Train LightGBM and return evaluation score. Higher is better."""
    if n_estimators is None:
        n_estimators = N_ESTIMATORS_INDIVIDUAL
    if max_depth is None:
        max_depth = MAX_DEPTH_INDIVIDUAL
    if X_train.shape[0] < 3 or X_val.shape[0] < 3 or X_train.shape[1] == 0:
        return float("nan")
    params = {
        "n_estimators": n_estimators, "max_depth": max_depth,
        "learning_rate": 0.1, "subsample": 0.8, "colsample_bytree": 0.8,
        "verbosity": -1, "n_jobs": 1, "random_state": seed,
    }
    try:
        if task_type == "binary_classification":
            model = lgb.LGBMClassifier(**params)
            model.fit(X_train, y_train, eval_set=[(X_val, y_val)])
            y_pred = model.predict_proba(X_val)
            if y_pred.ndim == 2:
                y_pred = y_pred[:, 1]
            if len(np.unique(y_val)) < 2:
                return float("nan")
            return roc_auc_score(y_val, y_pred)
        else:
            model = lgb.LGBMRegressor(**params)
            model.fit(X_train, y_train, eval_set=[(X_val, y_val)])
            y_pred = model.predict(X_val)
            mae = mean_absolute_error(y_val, y_pred)
            return 1.0 / (1.0 + mae)
    except Exception:
        return float("nan")


def compute_jrn_for_task_join(task_name, join_id, X_train_base, y_train,
                              X_val_base, y_val, task_type, tables,
                              table_pk_cols, entity_col, agg_funcs=None):
    """Compute entity-centric JRN for a single join on a single task."""
    if agg_funcs is None:
        agg_funcs = ["mean"]
    NaN_RESULT = {"jrn": float("nan"), "baseline": float("nan"), "augmented": float("nan")}
    fk_def = next((f for f in FK_DEFS if f["id"] == join_id), None)
    if fk_def is None:
        return NaN_RESULT

    source_df = tables.get(fk_def["source"])
    target_df = tables.get(fk_def["target"])
    if source_df is None:
        return NaN_RESULT

    grouped = _aggregate_to_entity(
        source_df=source_df, entity_col=entity_col, target_df=target_df,
        fk_col=fk_def["fk_col"], pk_col=fk_def["pk_col"],
        agg_prefix=f"{join_id}_{fk_def['source']}_{fk_def['target']}_",
        agg_funcs=agg_funcs, tables=tables, table_pk_cols=table_pk_cols,
        source_table_name=fk_def["source"],
    )
    if grouped is None or grouped.empty:
        return NaN_RESULT
    if entity_col not in X_train_base.columns:
        return NaN_RESULT

    X_train_aug = X_train_base.merge(grouped, on=entity_col, how="left")
    X_val_aug = X_val_base.merge(grouped, on=entity_col, how="left")
    X_train_aug = _make_numeric_df(X_train_aug)
    X_val_aug = _make_numeric_df(X_val_aug)

    agg_cols = [c for c in X_train_aug.columns if c.startswith("agg_")]
    if not agg_cols:
        return NaN_RESULT

    common_aug = sorted(set(X_train_aug.columns) & set(X_val_aug.columns))
    X_train_aug = X_train_aug[common_aug]
    X_val_aug = X_val_aug[common_aug]

    base_cols = sorted(set(X_train_base.columns) & set(X_val_base.columns))
    X_train_b = _make_numeric_df(X_train_base[base_cols])
    X_val_b = _make_numeric_df(X_val_base[base_cols])

    baseline_scores, augmented_scores = [], []
    for seed in SEEDS:
        sb = train_eval_lgbm(X_train_b, y_train, X_val_b, y_val, task_type, seed)
        sa = train_eval_lgbm(X_train_aug, y_train, X_val_aug, y_val, task_type, seed)
        if not math.isnan(sb) and not math.isnan(sa):
            baseline_scores.append(sb)
            augmented_scores.append(sa)

    if not baseline_scores:
        return NaN_RESULT
    mean_base = float(np.mean(baseline_scores))
    mean_aug = float(np.mean(augmented_scores))
    jrn = mean_aug / mean_base if mean_base > 1e-10 else float("nan")
    return {"jrn": jrn, "baseline": mean_base, "augmented": mean_aug}


def compute_local_jrn(join_id, tables, table_pk_cols):
    """Compute LOCAL JRN: predict a proxy column of the target table with/without source features."""
    NaN_RESULT = {"local_jrn": float("nan"), "proxy_col": None}
    fk_def = next((f for f in FK_DEFS if f["id"] == join_id), None)
    if fk_def is None:
        return NaN_RESULT

    source_df = tables.get(fk_def["source"])
    target_df = tables.get(fk_def["target"])
    pk_col = fk_def["pk_col"]
    fk_col = fk_def["fk_col"]
    if target_df is None or source_df is None:
        return NaN_RESULT

    target_enc = encode_table(target_df, pk_col)
    numeric_cols = target_enc.select_dtypes(include=[np.number]).columns.tolist()
    numeric_cols = [c for c in numeric_cols if c != pk_col]
    if len(numeric_cols) < 2:
        return NaN_RESULT

    variances = target_enc[numeric_cols].var()
    proxy_col = variances.idxmax()
    feature_cols = [c for c in numeric_cols if c != proxy_col]
    if not feature_cols:
        return NaN_RESULT

    y = target_enc[proxy_col]
    X_base = target_enc[feature_cols].fillna(0)

    source_enc = encode_table(source_df, fk_col)
    src_numeric = source_enc.select_dtypes(include=[np.number]).columns.tolist()
    src_numeric = [c for c in src_numeric if c != fk_col]
    if not src_numeric:
        return NaN_RESULT

    try:
        grouped = source_enc.groupby(fk_col)[src_numeric].mean().reset_index()
        grouped.columns = [fk_col] + [f"local_agg_{c}" for c in src_numeric]
        augmented = target_enc.merge(grouped, left_on=pk_col, right_on=fk_col, how="left")
        aug_cols = [c for c in augmented.columns if c.startswith("local_agg_")]
        X_aug = augmented[feature_cols + aug_cols].fillna(0)
    except Exception:
        return NaN_RESULT

    scores_base, scores_aug = [], []
    for seed in SEEDS:
        try:
            Xtr, Xte, ytr, yte = train_test_split(X_base, y, test_size=0.3, random_state=seed)
            Xtr_a, Xte_a = X_aug.loc[Xtr.index], X_aug.loc[Xte.index]
            sb = train_eval_lgbm(Xtr, ytr, Xte, yte, "regression", seed,
                                n_estimators=N_ESTIMATORS_LOCAL, max_depth=MAX_DEPTH_LOCAL)
            sa = train_eval_lgbm(Xtr_a, ytr, Xte_a, yte, "regression", seed,
                                n_estimators=N_ESTIMATORS_LOCAL, max_depth=MAX_DEPTH_LOCAL)
            if not math.isnan(sb) and not math.isnan(sa):
                scores_base.append(sb)
                scores_aug.append(sa)
        except Exception:
            continue

    if not scores_base:
        return NaN_RESULT
    mean_b = float(np.mean(scores_base))
    mean_a = float(np.mean(scores_aug))
    return {"local_jrn": mean_a / mean_b if mean_b > 1e-10 else float("nan"),
            "proxy_col": proxy_col, "baseline": mean_b, "augmented": mean_a}


print("JRN computation functions loaded.")

## Part A: Individual JRN Computation

Compute JRN for each FK join × task combination, plus local proxy JRN for all 13 joins.

In [ ]:
t_start = time.time()
print("Computing individual JRN for all joins x tasks...")

individual_jrn = {}
for task_name, td in task_datasets.items():
    print(f"\nTask: {task_name}")
    entity_col = td["entity_col"]
    for fk_def in FK_DEFS:
        join_id = fk_def["id"]
        t0 = time.time()
        result = compute_jrn_for_task_join(
            task_name=task_name, join_id=join_id,
            X_train_base=td["X_train"], y_train=td["y_train"],
            X_val_base=td["X_val"], y_val=td["y_val"],
            task_type=td["task_type"], tables=tables,
            table_pk_cols=table_pk_cols, entity_col=entity_col,
        )
        if join_id not in individual_jrn:
            individual_jrn[join_id] = {}
        individual_jrn[join_id][task_name] = result
        jrn_val = result.get("jrn", float("nan"))
        status = f"JRN={jrn_val:.4f}" if not math.isnan(jrn_val) else "JRN=NaN"
        print(f"  {join_id} ({fk_def['source']}->{fk_def['target']}): {status} [{time.time()-t0:.1f}s]")

print("\nComputing local JRN (proxy-target based)...")
local_jrn = {}
for fk_def in FK_DEFS:
    join_id = fk_def["id"]
    result = compute_local_jrn(join_id, tables, table_pk_cols)
    local_jrn[join_id] = result
    lj = result.get("local_jrn", float("nan"))
    status = f"local_JRN={lj:.4f}" if not math.isnan(lj) else "local_JRN=NaN"
    print(f"  {join_id}: {status}")

print(f"\nPart A individual JRN computed in {time.time()-t_start:.1f}s")

## Part A (cont.): Chain Compounding Test

Test the multiplicative compounding hypothesis: does chain JRN ≈ product of individual JRNs?

In [ ]:
def get_best_jrn(join_id, task_name=None):
    """Get best available JRN for a join."""
    if task_name and join_id in individual_jrn and task_name in individual_jrn[join_id]:
        v = individual_jrn[join_id][task_name].get("jrn", float("nan"))
        if not math.isnan(v):
            return v
    if join_id in individual_jrn:
        all_v = [individual_jrn[join_id][t].get("jrn", float("nan"))
                 for t in individual_jrn[join_id]
                 if not math.isnan(individual_jrn[join_id][t].get("jrn", float("nan")))]
        if all_v:
            return float(np.mean(all_v))
    if join_id in local_jrn:
        v = local_jrn[join_id].get("local_jrn", float("nan"))
        if not math.isnan(v):
            return v
    return float("nan")


def compute_chain_features(chain_def, tables, table_pk_cols, entity_col, entity_table, agg_funcs=None):
    """Compute aggregated features for a chain of joins."""
    if agg_funcs is None:
        agg_funcs = ["mean"]
    chain_type = chain_def["type"]
    join_ids = chain_def["joins"]

    if chain_type == "linear":
        fk_defs_chain = [next((f for f in FK_DEFS if f["id"] == jid), None) for jid in join_ids]
        if any(f is None for f in fk_defs_chain):
            return None
        first_fk = fk_defs_chain[0]
        current_df = tables.get(first_fk["source"])
        if current_df is None or entity_col not in current_df.columns:
            return None
        current_df = encode_table(current_df, entity_col)
        for fk_def_c in fk_defs_chain:
            target_df = tables.get(fk_def_c["target"])
            if target_df is None:
                return None
            target_enc = encode_table(target_df, fk_def_c["pk_col"])
            target_numeric = target_enc.select_dtypes(include=[np.number]).columns.tolist()
            target_numeric = [c for c in target_numeric if c != fk_def_c["pk_col"]]
            if not target_numeric:
                continue
            current_df = current_df.merge(
                target_enc[target_numeric + [fk_def_c["pk_col"]]],
                left_on=fk_def_c["fk_col"], right_on=fk_def_c["pk_col"],
                how="left", suffixes=("", f"_{fk_def_c['id']}")
            )
        numeric_cols = current_df.select_dtypes(include=[np.number]).columns.tolist()
        numeric_cols = [c for c in numeric_cols if c != entity_col]
        if not numeric_cols:
            return None
        grouped = current_df.groupby(entity_col)[numeric_cols].agg(agg_funcs)
        if isinstance(grouped.columns, pd.MultiIndex):
            grouped.columns = [f"chain_{chain_def['id']}_{col}_{func}" for col, func in grouped.columns]
        else:
            grouped.columns = [f"chain_{chain_def['id']}_{col}" for col in grouped.columns]
        return grouped.reset_index()

    elif chain_type == "convergent":
        fk_defs_chain = [next((f for f in FK_DEFS if f["id"] == jid), None) for jid in join_ids]
        if any(f is None for f in fk_defs_chain):
            return None
        all_grouped = None
        for fk_def_c in fk_defs_chain:
            source_df = tables.get(fk_def_c["source"])
            if source_df is None:
                continue
            source_enc = encode_table(source_df, fk_def_c["fk_col"])
            numeric_cols = source_enc.select_dtypes(include=[np.number]).columns.tolist()
            numeric_cols = [c for c in numeric_cols if c != fk_def_c["fk_col"] and c != entity_col]
            if not numeric_cols:
                continue
            if entity_col in source_enc.columns:
                grp_col = entity_col
            elif fk_def_c["fk_col"] == entity_col or fk_def_c["pk_col"] == entity_col:
                grp_col = fk_def_c["fk_col"]
            else:
                continue
            grouped = source_enc.groupby(grp_col)[numeric_cols].agg(agg_funcs)
            if isinstance(grouped.columns, pd.MultiIndex):
                grouped.columns = [f"chain_{chain_def['id']}_{fk_def_c['source']}_{col}_{func}" for col, func in grouped.columns]
            else:
                grouped.columns = [f"chain_{chain_def['id']}_{fk_def_c['source']}_{col}" for col in grouped.columns]
            grouped = grouped.reset_index()
            if grp_col != entity_col:
                grouped = grouped.rename(columns={grp_col: entity_col})
            if all_grouped is None:
                all_grouped = grouped
            else:
                all_grouped = all_grouped.merge(grouped, on=entity_col, how="outer")
        return all_grouped

    elif chain_type == "linear_3hop":
        results_df = tables.get("results")
        if results_df is None or entity_col not in results_df.columns:
            return None
        current_df = results_df[[entity_col, "raceId"]].copy()
        races_df = tables.get("races")
        if races_df is None:
            return None
        races_enc = encode_table(races_df, "raceId")
        current_df = current_df.merge(races_enc, on="raceId", how="left")
        circuits_df = tables.get("circuits")
        if circuits_df is None:
            return None
        circuits_enc = encode_table(circuits_df, "circuitId")
        circuit_numeric = circuits_enc.select_dtypes(include=[np.number]).columns.tolist()
        circuit_numeric = [c for c in circuit_numeric if c != "circuitId"]
        if "circuitId" in current_df.columns:
            current_df = current_df.merge(
                circuits_enc[circuit_numeric + ["circuitId"]],
                on="circuitId", how="left"
            )
        numeric_cols = current_df.select_dtypes(include=[np.number]).columns.tolist()
        numeric_cols = [c for c in numeric_cols if c != entity_col]
        if not numeric_cols:
            return None
        grouped = current_df.groupby(entity_col)[numeric_cols].agg(agg_funcs)
        if isinstance(grouped.columns, pd.MultiIndex):
            grouped.columns = [f"chain3hop_{col}_{func}" for col, func in grouped.columns]
        else:
            grouped.columns = [f"chain3hop_{col}" for col in grouped.columns]
        return grouped.reset_index()
    return None


print("Computing chain JRN (multiplicative compounding test)...")
chain_results = []

for chain_def in CHAINS:
    for task_name, td in task_datasets.items():
        entity_col = td["entity_col"]
        entity_table = task_meta[task_name].get("entity_table", "drivers")
        join_jrns = []
        for jid in chain_def["joins"]:
            v = get_best_jrn(jid, task_name)
            if not math.isnan(v):
                join_jrns.append(v)
        if len(join_jrns) != len(chain_def["joins"]):
            continue
        predicted_jrn = np.prod(join_jrns)
        chain_features = compute_chain_features(
            chain_def, tables, table_pk_cols, entity_col, entity_table
        )
        if chain_features is None or chain_features.empty:
            continue
        X_train_aug = td["X_train"].copy()
        X_val_aug = td["X_val"].copy()
        if entity_col in X_train_aug.columns and entity_col in chain_features.columns:
            X_train_aug = X_train_aug.merge(chain_features, on=entity_col, how="left")
            X_val_aug = X_val_aug.merge(chain_features, on=entity_col, how="left")
        else:
            continue
        X_train_aug = _make_numeric_df(X_train_aug.fillna(0))
        X_val_aug = _make_numeric_df(X_val_aug.fillna(0))
        common_cols = sorted(set(X_train_aug.columns) & set(X_val_aug.columns))
        X_train_aug = X_train_aug[common_cols]
        X_val_aug = X_val_aug[common_cols]

        base_cols = sorted(set(td["X_train"].columns) & set(td["X_val"].columns))
        base_cols = [c for c in base_cols if td["X_train"][c].dtype in [np.float64, np.float32, np.int64, np.int32]]

        measured_base, measured_aug = [], []
        for seed in SEEDS:
            sb = train_eval_lgbm(td["X_train"][base_cols], td["y_train"],
                                td["X_val"][base_cols], td["y_val"], td["task_type"], seed)
            sa = train_eval_lgbm(X_train_aug, td["y_train"],
                                X_val_aug, td["y_val"], td["task_type"], seed)
            if not math.isnan(sb) and not math.isnan(sa):
                measured_base.append(sb)
                measured_aug.append(sa)
        if not measured_base:
            continue
        mean_base = np.mean(measured_base)
        mean_aug = np.mean(measured_aug)
        if mean_base < 1e-10:
            continue
        measured_jrn = mean_aug / mean_base
        chain_results.append({
            "chain_id": chain_def["id"], "chain_desc": chain_def["desc"],
            "task": task_name, "predicted_jrn": float(predicted_jrn),
            "measured_jrn": float(measured_jrn),
        })
        print(f"  {chain_def['desc']} / {task_name.split('/')[-1]}: pred={predicted_jrn:.4f}, meas={measured_jrn:.4f}")

if len(chain_results) >= 3:
    predicted = np.array([r["predicted_jrn"] for r in chain_results])
    measured = np.array([r["measured_jrn"] for r in chain_results])
    mask = np.ones(len(predicted), dtype=bool)
    for arr in [predicted, measured]:
        mu, sigma = np.mean(arr), np.std(arr)
        if sigma > 0:
            mask &= np.abs(arr - mu) < 3 * sigma
    predicted_clean = predicted[mask]
    measured_clean = measured[mask]
    if len(predicted_clean) >= 3:
        r2 = r2_score(measured_clean, predicted_clean)
        spearman_rho_chain, _ = stats.spearmanr(predicted_clean, measured_clean)
        print(f"\nCompounding R\u00b2: {r2:.4f}, Spearman rho: {spearman_rho_chain:.4f} (n={len(predicted_clean)})")
    else:
        r2, spearman_rho_chain = float("nan"), float("nan")
else:
    r2, spearman_rho_chain = float("nan"), float("nan")
    print(f"Only {len(chain_results)} chain results, insufficient for R\u00b2")

## Part B: Entropy-Based JRN Proxy

Compute an entropy-based JRN proxy (H(Y) - H(Y|aggregated_features)) and compare with GBM probe rankings.

In [ ]:
def compute_entropy(values):
    """Compute Shannon entropy of a discrete array."""
    counts = np.bincount(values.astype(int))
    counts = counts[counts > 0]
    p = counts / counts.sum()
    return -np.sum(p * np.log2(p + 1e-12))


def compute_conditional_entropy(x, y):
    """Compute H(Y|X)."""
    unique_x = np.unique(x)
    h_y_given_x = 0.0
    n = len(y)
    for xv in unique_x:
        mask_x = x == xv
        p_x = mask_x.sum() / n
        if mask_x.sum() > 0:
            h_y_given_x += p_x * compute_entropy(y[mask_x])
    return h_y_given_x


def compute_entropy_jrn(entity_ids, labels, source_df, entity_col,
                        source_fk, task_type, n_bins=None):
    """Compute entropy-based JRN proxy."""
    if n_bins is None:
        n_bins = N_BINS_ENTROPY
    if task_type == "regression":
        try:
            y_disc = pd.cut(labels, bins=n_bins, labels=False)
            y_disc = np.nan_to_num(y_disc, nan=0).astype(int)
        except Exception:
            y_disc = np.zeros(len(labels), dtype=int)
    else:
        y_disc = labels.astype(int)

    h_y = compute_entropy(y_disc)
    if h_y < 1e-10:
        return 0.0

    source_enc = encode_table(source_df, source_fk)
    numeric_cols = source_enc.select_dtypes(include=[np.number]).columns.tolist()
    link_col = entity_col if entity_col in source_enc.columns else source_fk
    numeric_cols = [c for c in numeric_cols if c != link_col and c != entity_col]
    if not numeric_cols:
        return 0.0

    grouped = source_enc.groupby(link_col)[numeric_cols].mean().reset_index()
    entity_to_idx = {eid: i for i, eid in enumerate(entity_ids)}
    feature_matrix = np.zeros((len(entity_ids), len(numeric_cols)))
    for _, row in grouped.iterrows():
        eid = row[link_col]
        if eid in entity_to_idx:
            feature_matrix[entity_to_idx[eid]] = row[numeric_cols].values

    min_h_y_given_x = h_y
    for j in range(feature_matrix.shape[1]):
        col_vals = feature_matrix[:, j]
        if np.std(col_vals) < 1e-10:
            continue
        try:
            binned = pd.cut(col_vals, bins=min(n_bins, len(np.unique(col_vals))), labels=False)
            binned = np.nan_to_num(binned, nan=0).astype(int)
        except Exception:
            continue
        h_cond = compute_conditional_entropy(binned, y_disc)
        min_h_y_given_x = min(min_h_y_given_x, h_cond)

    return max(0.0, h_y - min_h_y_given_x)


print("Computing entropy-based JRN proxy...")
entropy_values = {}
probe_jrn_values = {}

for task_name, td in task_datasets.items():
    entity_col = td["entity_col"]
    if entity_col not in td["X_train"].columns:
        continue
    entity_ids = td["X_train"][entity_col].values
    labels = td["y_train"].values

    for fk_def in FK_DEFS:
        join_id = fk_def["id"]
        source_df = tables.get(fk_def["source"])
        if source_df is None or entity_col not in source_df.columns:
            continue
        try:
            entropy_red = compute_entropy_jrn(
                entity_ids, labels, source_df, entity_col,
                fk_def["fk_col"], td["task_type"]
            )
        except Exception:
            entropy_red = 0.0

        if join_id not in entropy_values:
            entropy_values[join_id] = {}
        entropy_values[join_id][task_name] = float(entropy_red)

        if join_id not in probe_jrn_values:
            probe_jrn_values[join_id] = {}
        if join_id in individual_jrn and task_name in individual_jrn[join_id]:
            probe_jrn_values[join_id][task_name] = individual_jrn[join_id][task_name].get("jrn", float("nan"))

all_entropy, all_probe = [], []
for task_name in ALL_TASKS:
    for join_id in sorted(entropy_values.keys()):
        if task_name in entropy_values.get(join_id, {}):
            ev = entropy_values[join_id][task_name]
            pv = probe_jrn_values.get(join_id, {}).get(task_name, float("nan"))
            if not math.isnan(ev) and not math.isnan(pv):
                all_entropy.append(ev)
                all_probe.append(pv)

if len(all_entropy) >= 3:
    overall_rho, overall_p = stats.spearmanr(all_entropy, all_probe)
    print(f"Entropy vs Probe overall Spearman rho: {overall_rho:.4f} (n={len(all_entropy)})")
else:
    overall_rho = float("nan")
    print("Insufficient data for Spearman correlation")

## Part B (cont.): Architecture Comparison

Evaluate 4 architecture configurations:
1. **probe_guided** — include only HIGH/CRITICAL joins (by GBM probe)
2. **entropy_guided** — include only HIGH/CRITICAL joins (by entropy)
3. **uniform_mean** — include all joins with mean aggregation
4. **uniform_rich** — include all joins with rich aggregation (mean, std, max, min)

In [ ]:
def classify_joins(jrn_values, tasks):
    """Classify joins into HIGH/CRITICAL/LOW based on average JRN."""
    avg_jrn = {}
    for join_id, task_vals in jrn_values.items():
        vals = [task_vals[t] for t in tasks if t in task_vals
                and not math.isnan(task_vals.get(t, float("nan")))]
        if vals:
            avg_jrn[join_id] = np.mean(vals)
    if not avg_jrn:
        return {}
    values = list(avg_jrn.values())
    if len(values) < 3:
        return {jid: "HIGH" for jid in avg_jrn}
    p33, p67 = np.percentile(values, 33), np.percentile(values, 67)
    return {jid: "HIGH" if v >= p67 else "CRITICAL" if v >= p33 else "LOW"
            for jid, v in avg_jrn.items()}


def build_config_features(config_name, join_classifications, X_base, entity_col,
                          tables, table_pk_cols):
    """Build feature matrix for a given architecture configuration."""
    X = X_base.copy()
    for fk_def in FK_DEFS:
        join_id = fk_def["id"]
        source_df = tables.get(fk_def["source"])
        target_df = tables.get(fk_def["target"])
        if source_df is None:
            continue
        category = join_classifications.get(join_id, "LOW")
        if config_name in ("probe_guided", "entropy_guided"):
            if category == "LOW":
                continue
            agg_funcs = ["mean"] if category == "HIGH" else ["mean", "std", "max", "min"]
        elif config_name == "uniform_mean":
            agg_funcs = ["mean"]
        elif config_name == "uniform_rich":
            agg_funcs = ["mean", "std", "max", "min"]
        else:
            agg_funcs = ["mean"]
        try:
            grouped = _aggregate_to_entity(
                source_df=source_df, entity_col=entity_col, target_df=target_df,
                fk_col=fk_def["fk_col"], pk_col=fk_def["pk_col"],
                agg_prefix=f"cfg_{join_id}_", agg_funcs=agg_funcs,
                tables=tables, table_pk_cols=table_pk_cols,
                source_table_name=fk_def["source"],
            )
            if grouped is not None and not grouped.empty and entity_col in X.columns:
                X = X.merge(grouped, on=entity_col, how="left")
        except Exception:
            continue
    return _make_numeric_df(X)


probe_classifications = classify_joins(probe_jrn_values, ALL_TASKS)
entropy_classifications = classify_joins(entropy_values, ALL_TASKS)
print(f"Probe classifications: {probe_classifications}")
print(f"Entropy classifications: {entropy_classifications}")

configs = {
    "probe_guided": probe_classifications,
    "entropy_guided": entropy_classifications,
    "uniform_mean": {fk["id"]: "HIGH" for fk in FK_DEFS},
    "uniform_rich": {fk["id"]: "CRITICAL" for fk in FK_DEFS},
}

config_results = {}
for task_name, td in task_datasets.items():
    entity_col = td["entity_col"]
    print(f"\nTask: {task_name}")
    config_results[task_name] = {}
    for config_name, join_class in configs.items():
        t0 = time.time()
        X_train_cfg = build_config_features(
            config_name, join_class, td["X_train"], entity_col, tables, table_pk_cols
        )
        X_val_cfg = build_config_features(
            config_name, join_class, td["X_val"], entity_col, tables, table_pk_cols
        )
        common = sorted(set(X_train_cfg.columns) & set(X_val_cfg.columns))
        numeric = [c for c in common if X_train_cfg[c].dtype in [np.float64, np.float32, np.int64, np.int32]]
        X_train_cfg = X_train_cfg[numeric].fillna(0)
        X_val_cfg = X_val_cfg[numeric].fillna(0)

        scores = []
        for seed in SEEDS:
            s = train_eval_lgbm(X_train_cfg, td["y_train"], X_val_cfg, td["y_val"],
                               td["task_type"], seed, n_estimators=N_ESTIMATORS_ARCH,
                               max_depth=MAX_DEPTH_ARCH)
            if not math.isnan(s):
                scores.append(s)

        mean_score = float(np.mean(scores)) if scores else float("nan")
        config_results[task_name][config_name] = {
            "mean": mean_score, "n_features": X_train_cfg.shape[1],
        }
        print(f"  {config_name}: {mean_score:.4f} (n_feats={X_train_cfg.shape[1]}) [{time.time()-t0:.1f}s]")

wins = {c: 0 for c in configs}
for task_name, task_cfg in config_results.items():
    best = max(task_cfg, key=lambda c: task_cfg[c]["mean"] if not math.isnan(task_cfg[c]["mean"]) else -999)
    wins[best] += 1
print(f"\nArchitecture wins: {wins}")

## Results Visualization

Summary tables and plots of the key findings from both Part A and Part B.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# ── Plot 1: Individual JRN heatmap ──────────────────────────────────────
ax = axes[0]
jrn_matrix = []
join_ids_present = sorted(individual_jrn.keys())
tasks_present = [t for t in ALL_TASKS if t in task_datasets]
for join_id in join_ids_present:
    row = []
    for task_name in tasks_present:
        v = individual_jrn.get(join_id, {}).get(task_name, {}).get("jrn", float("nan"))
        row.append(v if not math.isnan(v) else 0)
    jrn_matrix.append(row)
jrn_arr = np.array(jrn_matrix)
if jrn_arr.size > 0:
    im = ax.imshow(jrn_arr, cmap='RdYlGn', aspect='auto', vmin=0.8, vmax=1.3)
    ax.set_yticks(range(len(join_ids_present)))
    ax.set_yticklabels(join_ids_present, fontsize=8)
    ax.set_xticks(range(len(tasks_present)))
    ax.set_xticklabels([t.split('/')[-1] for t in tasks_present], rotation=45, ha='right', fontsize=8)
    ax.set_title('Individual JRN per Join x Task', fontsize=10)
    plt.colorbar(im, ax=ax, shrink=0.8)

# ── Plot 2: Predicted vs Measured chain JRN ────────────────────────────
ax = axes[1]
if chain_results:
    pred = [r["predicted_jrn"] for r in chain_results]
    meas = [r["measured_jrn"] for r in chain_results]
    ax.scatter(pred, meas, alpha=0.7, edgecolors='k', s=60)
    all_vals = pred + meas
    lims = [min(all_vals) * 0.95, max(all_vals) * 1.05]
    ax.plot(lims, lims, 'r--', alpha=0.5, label='y=x (perfect compounding)')
    ax.set_xlabel('Predicted JRN (product)', fontsize=9)
    ax.set_ylabel('Measured JRN', fontsize=9)
    r2_label = f'R\u00b2={r2:.3f}' if not math.isnan(r2) else 'R\u00b2=N/A'
    ax.set_title(f'Chain Compounding ({r2_label})', fontsize=10)
    ax.legend(fontsize=8)
else:
    ax.text(0.5, 0.5, 'No chain results', ha='center', va='center', transform=ax.transAxes)
    ax.set_title('Chain Compounding', fontsize=10)

# ── Plot 3: Architecture comparison ────────────────────────────────────
ax = axes[2]
if config_results:
    config_names = list(next(iter(config_results.values())).keys())
    x = np.arange(len(tasks_present))
    width = 0.2
    colors = ['#2196F3', '#4CAF50', '#FF9800', '#E91E63']
    for i, cfg in enumerate(config_names):
        vals = [config_results.get(t, {}).get(cfg, {}).get("mean", 0) for t in tasks_present]
        ax.bar(x + i * width, vals, width, label=cfg, color=colors[i % len(colors)], alpha=0.8)
    ax.set_xticks(x + width * 1.5)
    ax.set_xticklabels([t.split('/')[-1] for t in tasks_present], rotation=45, ha='right', fontsize=8)
    ax.set_ylabel('Score', fontsize=9)
    ax.set_title('Architecture Comparison', fontsize=10)
    ax.legend(fontsize=7, loc='upper right')
else:
    ax.text(0.5, 0.5, 'No config results', ha='center', va='center', transform=ax.transAxes)

plt.tight_layout()
plt.savefig('jrn_results.png', dpi=150, bbox_inches='tight')
plt.show()

# ── Summary table ──────────────────────────────────────────────────────
print("\n" + "=" * 70)
print("EXPERIMENT SUMMARY")
print("=" * 70)
print(f"\nPart A - Multiplicative Compounding:")
print(f"  Chains evaluated: {len(chain_results)}")
if not math.isnan(r2):
    print(f"  R\u00b2 (predicted vs measured): {r2:.4f}")
else:
    print(f"  R\u00b2: N/A (insufficient chain data)")
if not math.isnan(spearman_rho_chain):
    print(f"  Spearman rho: {spearman_rho_chain:.4f}")
print(f"\nPart B - Entropy vs Probe:")
if not math.isnan(overall_rho):
    print(f"  Overall Spearman rho: {overall_rho:.4f}")
else:
    print(f"  Overall Spearman rho: N/A")
print(f"\nPart B - Architecture Wins:")
for cfg, n in wins.items():
    print(f"  {cfg}: {n} wins")
print(f"\nBest config per task:")
for task_name, task_cfg in config_results.items():
    best = max(task_cfg, key=lambda c: task_cfg[c]["mean"] if not math.isnan(task_cfg[c]["mean"]) else -999)
    print(f"  {task_name.split('/')[-1]}: {best} ({task_cfg[best]['mean']:.4f})")